# PDF Table Extractor - Interactive Notebook

This notebook demonstrates how to extract all tables from a multi-page PDF and combine them into a single structured output. We'll ensure continuity across pages and preserve table formatting for export to CSV, Excel, or HTML.

## Features
- Extract tables from multi-page PDFs
- Detect and handle table continuity across pages
- Clean and structure table data
- Export to multiple formats (CSV, Excel, HTML)
- Interactive data exploration and validation

## 1. Install and Import Required Libraries

First, let's install and import all the necessary libraries for PDF processing and data manipulation.

In [ ]:
# Install required packages (run this only once)
!pip install tabula-py pandas openpyxl camelot-py[cv] pdfplumber beautifulsoup4 streamlit reportlab

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import io
import warnings
warnings.filterwarnings('ignore')

# PDF processing libraries
import tabula
import camelot
import pdfplumber
from bs4 import BeautifulSoup

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")

## 2. Load and Analyze PDF Document

Let's start by creating a sample PDF with tables or loading an existing PDF file to analyze its structure.

In [ ]:
# Create a sample PDF with tables for demonstration
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def create_sample_pdf():
    """Create a sample multi-page PDF with tables for testing."""
    doc = SimpleDocTemplate("sample_multipage_tables.pdf", pagesize=letter)
    elements = []
    styles = getSampleStyleSheet()
    
    # Page 1 - Sales Data (Part 1)
    elements.append(Paragraph("Quarterly Sales Report", styles['Title']))
    elements.append(Spacer(1, 20))
    
    elements.append(Paragraph("Q1 Sales Data", styles['Heading2']))
    sales_data_p1 = [
        ['Product', 'Region', 'Q1 Sales', 'Units Sold', 'Revenue'],
        ['Laptop Pro', 'North', '$125,000', '50', '$2,500,000'],
        ['Tablet Max', 'North', '$89,000', '178', '$1,780,000'],
        ['Phone Elite', 'North', '$156,000', '312', '$3,120,000'],
        ['Laptop Pro', 'South', '$98,000', '39', '$1,950,000'],
        ['Tablet Max', 'South', '$67,000', '134', '$1,340,000']
    ]
    
    table1 = Table(sales_data_p1)
    table1.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 12),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.lightgrey),
        ('GRID', (0, 0), (-1, -1), 1, colors.black)
    ]))
    elements.append(table1)
    
    # Add some text and page break
    elements.append(Spacer(1, 30))
    elements.append(Paragraph("Note: Sales data continues on next page", styles['Normal']))
    elements.append(PageBreak())
    
    # Page 2 - Sales Data (Part 2 - Continuation)
    elements.append(Paragraph("Quarterly Sales Report (Continued)", styles['Title']))
    elements.append(Spacer(1, 20))
    
    sales_data_p2 = [
        ['Product', 'Region', 'Q1 Sales', 'Units Sold', 'Revenue'],  # Header repeated
        ['Phone Elite', 'South', '$123,000', '246', '$2,460,000'],
        ['Laptop Pro', 'East', '$110,000', '44', '$2,200,000'],
        ['Tablet Max', 'East', '$78,000', '156', '$1,560,000'],
        ['Phone Elite', 'East', '$145,000', '290', '$2,900,000'],
        ['Laptop Pro', 'West', '$87,000', '35', '$1,750,000']
    ]
    
    table2 = Table(sales_data_p2)
    table2.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 12),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.lightgrey),
        ('GRID', (0, 0), (-1, -1), 1, colors.black)
    ]))
    elements.append(table2)
    
    elements.append(Spacer(1, 30))
    
    # Page 2 - Different table (Expense Report)
    elements.append(Paragraph("Monthly Expense Report", styles['Heading2']))
    expense_data = [
        ['Category', 'January', 'February', 'March', 'Total'],
        ['Office Supplies', '$2,500', '$3,200', '$2,800', '$8,500'],
        ['Marketing', '$15,000', '$18,000', '$16,500', '$49,500'],
        ['Travel', '$8,000', '$6,500', '$9,200', '$23,700'],
        ['Utilities', '$3,500', '$3,800', '$3,600', '$10,900']
    ]
    
    table3 = Table(expense_data)
    table3.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.darkgreen),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 12),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.lightgreen),
        ('GRID', (0, 0), (-1, -1), 1, colors.black)
    ]))
    elements.append(table3)
    
    # Build the PDF
    doc.build(elements)
    print("✅ Created sample_multipage_tables.pdf")
    return "sample_multipage_tables.pdf"

# Create sample PDF
sample_pdf_path = create_sample_pdf()

# Analyze PDF structure
def analyze_pdf_structure(pdf_path):
    """Analyze the structure of a PDF file."""
    with pdfplumber.open(pdf_path) as pdf:
        print(f"📄 PDF Analysis: {pdf_path}")
        print(f"   Total pages: {len(pdf.pages)}")
        print(f"   PDF info: {pdf.metadata}")
        
        for i, page in enumerate(pdf.pages, 1):
            print(f"\n📋 Page {i}:")
            print(f"   Size: {page.width:.1f} x {page.height:.1f} points")
            
            # Count tables on this page
            tables = page.extract_tables()
            print(f"   Tables found: {len(tables)}")
            
            for j, table in enumerate(tables, 1):
                if table:
                    print(f"     Table {j}: {len(table)} rows x {len(table[0])} columns")

analyze_pdf_structure(sample_pdf_path)

## 3. Extract Tables from All Pages

Now let's extract tables using multiple methods to ensure we capture all tables accurately.

In [ ]:
# Import our custom extractor
from pdf_table_extractor import PDFTableExtractor, TableInfo

# Create extractor instance
extractor = PDFTableExtractor(sample_pdf_path, debug=True)

# Extract tables using different methods
print("🔍 Extracting tables using multiple methods...\n")

# Method 1: pdfplumber
pdfplumber_tables = extractor.extract_with_pdfplumber()
print(f"pdfplumber found: {len(pdfplumber_tables)} tables")

# Method 2: camelot
try:
    camelot_tables = extractor.extract_with_camelot()
    print(f"camelot found: {len(camelot_tables)} tables")
except Exception as e:
    print(f"camelot error: {e}")
    camelot_tables = []

# Method 3: tabula
try:
    tabula_tables = extractor.extract_with_tabula()
    print(f"tabula found: {len(tabula_tables)} tables")
except Exception as e:
    print(f"tabula error: {e}")
    tabula_tables = []

# Combine results
all_extracted_tables = pdfplumber_tables + camelot_tables + tabula_tables
print(f"\n📊 Total tables extracted: {len(all_extracted_tables)}")

In [ ]:
# Display extraction results
print("📋 Extraction Results Summary:\n")

for i, table_info in enumerate(all_extracted_tables, 1):
    print(f"Table {i}:")
    print(f"  Page: {table_info.page_number}")
    print(f"  Method: {table_info.method}")
    print(f"  Shape: {table_info.data.shape}")
    print(f"  Confidence: {table_info.confidence:.2f}")
    print(f"  Preview:")
    display(table_info.data.head(3))
    print("-" * 50)

## 4. Clean and Structure Table Data

Let's clean the extracted table data by removing empty rows, handling merged cells, and standardizing column headers.

In [ ]:
def clean_and_structure_table(df):
    """Clean and structure a single table DataFrame."""
    print(f"Original shape: {df.shape}")
    
    # Remove completely empty rows and columns
    df_clean = df.dropna(how='all').dropna(axis=1, how='all')
    print(f"After removing empty rows/cols: {df_clean.shape}")
    
    # Clean whitespace in string columns
    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            df_clean[col] = df_clean[col].astype(str).str.strip()
            # Replace 'nan' strings with actual NaN
            df_clean[col] = df_clean[col].replace(['nan', 'None', ''], np.nan)
    
    # Check if first row should be header
    if not df_clean.empty:
        first_row = df_clean.iloc[0]
        # If first row contains mostly text and subsequent rows contain numbers/mixed data
        if (first_row.astype(str).str.match(r'^[A-Za-z\s]+$').sum() > len(first_row) * 0.7):
            df_clean.columns = first_row
            df_clean = df_clean.iloc[1:].reset_index(drop=True)
            print("✅ Set first row as column headers")
    
    print(f"Final cleaned shape: {df_clean.shape}")
    return df_clean

# Clean all extracted tables
cleaned_tables = []
for i, table_info in enumerate(all_extracted_tables, 1):
    print(f"\n🧹 Cleaning Table {i} (from {table_info.method}):")
    cleaned_df = clean_and_structure_table(table_info.data.copy())
    
    if not cleaned_df.empty:
        cleaned_tables.append({
            'table': cleaned_df,
            'page': table_info.page_number,
            'method': table_info.method,
            'confidence': table_info.confidence
        })
    
    print("Column names:", list(cleaned_df.columns))

print(f"\n✅ Successfully cleaned {len(cleaned_tables)} tables")

## 5. Handle Table Continuity Across Pages

Let's identify and merge tables that span multiple pages by detecting header patterns and maintaining proper row sequencing.

In [ ]:
def detect_table_continuity(tables_info):
    """Detect which tables are continuations of each other."""
    if len(tables_info) <= 1:
        return [tables_info]
    
    # Sort by page number
    sorted_tables = sorted(tables_info, key=lambda x: x['page'])
    
    groups = []
    current_group = [sorted_tables[0]]
    
    for i in range(1, len(sorted_tables)):
        current = sorted_tables[i]
        previous = sorted_tables[i-1]
        
        # Check for table continuity
        current_cols = set(current['table'].columns)
        previous_cols = set(previous['table'].columns)
        
        # Calculate column similarity
        if len(current_cols) > 0 and len(previous_cols) > 0:
            similarity = len(current_cols.intersection(previous_cols)) / len(current_cols.union(previous_cols))
        else:
            similarity = 0
        
        # Criteria for continuation:
        # 1. High column similarity (>80%)
        # 2. Consecutive pages OR same page
        is_continuation = (
            similarity > 0.8 and 
            (current['page'] == previous['page'] + 1 or current['page'] == previous['page'])
        )
        
        print(f"Table {i+1} vs Table {i}: similarity={similarity:.2f}, is_continuation={is_continuation}")
        
        if is_continuation:
            current_group.append(current)
        else:
            groups.append(current_group)
            current_group = [current]
    
    groups.append(current_group)
    return groups

# Detect table groups
table_groups = detect_table_continuity(cleaned_tables)

print(f"\n🔗 Detected {len(table_groups)} table groups:")
for i, group in enumerate(table_groups, 1):
    pages = [t['page'] for t in group]
    methods = [t['method'] for t in group]
    print(f"  Group {i}: {len(group)} tables from pages {pages} (methods: {set(methods)})")

In [ ]:
def combine_table_group(group):
    """Combine a group of related tables into a single DataFrame."""
    if len(group) == 1:
        return group[0]['table']
    
    print(f"\n🔄 Combining {len(group)} tables...")
    
    # Use the table with highest confidence as the master
    master_table = max(group, key=lambda x: x['confidence'])
    master_columns = master_table['table'].columns.tolist()
    
    combined_data = []
    
    for i, table_info in enumerate(group):
        df = table_info['table'].copy()
        
        print(f"  Processing table from page {table_info['page']} ({table_info['method']})")
        print(f"    Shape: {df.shape}")
        
        # Align columns with master table
        if list(df.columns) != master_columns:
            if len(df.columns) == len(master_columns):
                df.columns = master_columns
                print(f"    ✅ Aligned columns with master table")
            else:
                print(f"    ⚠️  Column count mismatch: {len(df.columns)} vs {len(master_columns)}")
        
        # Skip header row if it's a repeat (except for first table)
        if i > 0 and not df.empty:
            first_row_str = df.iloc[0].astype(str).str.lower()
            master_cols_str = [str(col).lower() for col in master_columns]
            
            # Check if first row looks like headers
            if any(col in ' '.join(first_row_str) for col in master_cols_str):
                df = df.iloc[1:]  # Skip header row
                print(f"    ✅ Skipped duplicate header row")
        
        if not df.empty:
            combined_data.append(df)
    
    # Combine all DataFrames
    if combined_data:
        result = pd.concat(combined_data, ignore_index=True)
        print(f"  ✅ Combined into shape: {result.shape}")
        return result
    else:
        return pd.DataFrame()

# Combine table groups
final_tables = []
for i, group in enumerate(table_groups, 1):
    print(f"\n📊 Processing Table Group {i}:")
    combined_table = combine_table_group(group)
    
    if not combined_table.empty:
        final_tables.append(combined_table)
        print(f"Final table shape: {combined_table.shape}")

print(f"\n🎉 Successfully created {len(final_tables)} final tables!")

## 6. Combine Tables into Single Dataset

Let's examine our final tables and optionally combine them into a single comprehensive dataset.

In [ ]:
# Display final tables
for i, table in enumerate(final_tables, 1):
    print(f"\n📊 Final Table {i}:")
    print(f"Shape: {table.shape}")
    print(f"Columns: {list(table.columns)}")
    print("\nData preview:")
    display(table.head())
    
    # Show data types
    print("\nData types:")
    print(table.dtypes)
    
    # Show basic statistics for numeric columns
    numeric_cols = table.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print("\nNumeric column statistics:")
        display(table[numeric_cols].describe())
    
    print("=" * 80)

In [ ]:
# Optional: Combine all tables into a single dataset if they have compatible structures
def can_combine_tables(tables):
    """Check if tables can be combined based on column compatibility."""
    if len(tables) <= 1:
        return True, "Single table or empty"
    
    first_cols = set(tables[0].columns)
    
    for i, table in enumerate(tables[1:], 2):
        current_cols = set(table.columns)
        similarity = len(first_cols.intersection(current_cols)) / len(first_cols.union(current_cols))
        
        if similarity < 0.6:  # Less than 60% column similarity
            return False, f"Table {i} has low column similarity ({similarity:.2f})"
    
    return True, "All tables have compatible column structures"

# Check if we can combine all tables
can_combine, reason = can_combine_tables(final_tables)
print(f"Can combine all tables: {can_combine}")
print(f"Reason: {reason}")

if can_combine and len(final_tables) > 1:
    print("\n🔄 Combining all tables into single dataset...")
    
    # Add source table identifier
    for i, table in enumerate(final_tables):
        table['source_table'] = f"Table_{i+1}"
    
    # Combine all tables
    master_dataset = pd.concat(final_tables, ignore_index=True, sort=False)
    
    print(f"✅ Created master dataset with shape: {master_dataset.shape}")
    print(f"Columns: {list(master_dataset.columns)}")
    
    # Display combined dataset
    display(master_dataset.head(10))
    
else:
    print("\n📊 Tables will be kept separate due to structural differences")
    master_dataset = None

## 7. Export to Multiple Formats

Now let's export our extracted and processed tables to CSV, Excel, and HTML formats with proper formatting preservation.

In [ ]:
# Export functions
def export_to_csv(tables, output_prefix="extracted_tables"):
    """Export tables to CSV format."""
    csv_files = []
    
    if len(tables) == 1:
        filename = f"{output_prefix}.csv"
        tables[0].to_csv(filename, index=False)
        csv_files.append(filename)
        print(f"✅ Exported to {filename}")
    else:
        for i, table in enumerate(tables, 1):
            filename = f"{output_prefix}_table_{i}.csv"
            table.to_csv(filename, index=False)
            csv_files.append(filename)
            print(f"✅ Exported Table {i} to {filename}")
    
    return csv_files

def export_to_excel(tables, output_filename="extracted_tables.xlsx"):
    """Export tables to Excel with multiple sheets."""
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        for i, table in enumerate(tables, 1):
            sheet_name = f'Table_{i}' if len(tables) > 1 else 'Combined_Table'
            table.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Auto-adjust column widths
            worksheet = writer.sheets[sheet_name]
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
    
    print(f"✅ Exported {len(tables)} tables to {output_filename}")
    return output_filename

def export_to_html(tables, output_filename="extracted_tables.html"):
    """Export tables to styled HTML."""
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Extracted PDF Tables</title>
        <style>
            body { 
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; 
                margin: 40px; 
                background-color: #f5f5f5;
            }
            .container { 
                max-width: 1200px; 
                margin: 0 auto; 
                background-color: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            }
            h1 { color: #2c3e50; text-align: center; }
            h2 { 
                color: #34495e; 
                border-bottom: 2px solid #3498db;
                padding-bottom: 10px;
                margin-top: 40px;
            }
            table { 
                border-collapse: collapse; 
                width: 100%; 
                margin-bottom: 30px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
            }
            th { 
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                font-weight: bold;
                padding: 12px;
                text-align: left;
            }
            td { 
                border: 1px solid #ddd; 
                padding: 10px; 
                text-align: left;
            }
            tr:nth-child(even) { background-color: #f8f9fa; }
            tr:hover { background-color: #e3f2fd; }
            .metadata { 
                background-color: #ecf0f1;
                padding: 10px;
                border-radius: 5px;
                margin-bottom: 15px;
                font-size: 0.9em;
                color: #666;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>📊 Extracted PDF Tables</h1>
            <div class="metadata">
                <strong>Source:</strong> {pdf_name}<br>
                <strong>Extraction Date:</strong> {date}<br>
                <strong>Total Tables:</strong> {table_count}
            </div>
    """.format(
        pdf_name=sample_pdf_path,
        date=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        table_count=len(tables)
    )
    
    for i, table in enumerate(tables, 1):
        title = f"Table {i}" if len(tables) > 1 else "Combined Table"
        html_content += f"""
            <h2>{title}</h2>
            <div class="metadata">
                Rows: {len(table)} | Columns: {len(table.columns)} | Non-null values: {table.count().sum()}
            </div>
            {table.to_html(index=False, escape=False, classes='table')}
        """
    
    html_content += """
        </div>
    </body>
    </html>
    """
    
    # Save HTML file
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"✅ Exported {len(tables)} tables to {output_filename}")
    return output_filename

# Export all tables to different formats
print("💾 Exporting tables to multiple formats...\n")

# CSV export
csv_files = export_to_csv(final_tables)

# Excel export
excel_file = export_to_excel(final_tables)

# HTML export
html_file = export_to_html(final_tables)

print(f"\n📁 Export Summary:")
print(f"   CSV files: {len(csv_files)} files")
print(f"   Excel file: {excel_file}")
print(f"   HTML file: {html_file}")

## 8. Validate and Display Results

Let's validate our extracted data for completeness and accuracy, then display summary statistics and visualizations.

In [ ]:
# Data validation and quality assessment
def validate_table_data(table, table_name):
    """Validate extracted table data quality."""
    print(f"\n🔍 Validating {table_name}:")
    
    # Basic metrics
    total_cells = table.shape[0] * table.shape[1]
    non_null_cells = table.count().sum()
    completeness = (non_null_cells / total_cells) * 100
    
    print(f"   📊 Shape: {table.shape}")
    print(f"   📈 Data completeness: {completeness:.1f}%")
    print(f"   🔢 Non-null cells: {non_null_cells}/{total_cells}")
    
    # Column analysis
    print(f"   📋 Columns ({len(table.columns)}):")
    for col in table.columns:
        non_null_count = table[col].count()
        data_type = str(table[col].dtype)
        unique_count = table[col].nunique()
        print(f"      {col}: {non_null_count} values, {unique_count} unique, type: {data_type}")
    
    # Check for potential issues
    issues = []
    
    if completeness < 50:
        issues.append(f"Low data completeness ({completeness:.1f}%)")
    
    # Check for columns with all same values
    for col in table.columns:
        if table[col].nunique() == 1 and table[col].count() > 1:
            issues.append(f"Column '{col}' has identical values")
    
    # Check for suspiciously wide tables (might indicate extraction errors)
    if table.shape[1] > 20:
        issues.append(f"Very wide table ({table.shape[1]} columns) - check extraction quality")
    
    if issues:
        print(f"   ⚠️  Potential issues: {'; '.join(issues)}")
    else:
        print(f"   ✅ No issues detected")
    
    return completeness, issues

# Validate all tables
validation_results = []
for i, table in enumerate(final_tables, 1):
    completeness, issues = validate_table_data(table, f"Table {i}")
    validation_results.append({
        'table_name': f"Table {i}",
        'completeness': completeness,
        'issues': len(issues)
    })

# Create validation summary
validation_df = pd.DataFrame(validation_results)
print("\n📊 Validation Summary:")
display(validation_df)

In [ ]:
# Visualize extraction results
if final_tables:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('PDF Table Extraction Analysis', fontsize=16, fontweight='bold')
    
    # 1. Table sizes
    table_sizes = [table.shape[0] * table.shape[1] for table in final_tables]
    table_names = [f"Table {i}" for i in range(1, len(final_tables) + 1)]
    
    axes[0, 0].bar(table_names, table_sizes, color='skyblue')
    axes[0, 0].set_title('Table Sizes (Total Cells)')
    axes[0, 0].set_ylabel('Number of Cells')
    
    # 2. Data completeness
    completeness_scores = [result['completeness'] for result in validation_results]
    axes[0, 1].bar(table_names, completeness_scores, color='lightgreen')
    axes[0, 1].set_title('Data Completeness (%)')
    axes[0, 1].set_ylabel('Completeness %')
    axes[0, 1].set_ylim(0, 100)
    
    # 3. Column count distribution
    column_counts = [table.shape[1] for table in final_tables]
    axes[1, 0].bar(table_names, column_counts, color='coral')
    axes[1, 0].set_title('Number of Columns per Table')
    axes[1, 0].set_ylabel('Column Count')
    
    # 4. Row count distribution
    row_counts = [table.shape[0] for table in final_tables]
    axes[1, 1].bar(table_names, row_counts, color='gold')
    axes[1, 1].set_title('Number of Rows per Table')
    axes[1, 1].set_ylabel('Row Count')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n📈 Extraction Statistics:")
    print(f"   Total tables extracted: {len(final_tables)}")
    print(f"   Total rows: {sum(row_counts)}")
    print(f"   Total columns: {sum(column_counts)}")
    print(f"   Average completeness: {np.mean(completeness_scores):.1f}%")
    print(f"   Largest table: {max(row_counts)} rows x {max(column_counts)} columns")
    print(f"   Smallest table: {min(row_counts)} rows x {min(column_counts)} columns")

In [ ]:
# Interactive data exploration
print("🔍 Interactive Data Exploration:\n")

# If we have a master dataset, explore it
if 'master_dataset' in locals() and master_dataset is not None:
    print("📊 Master Dataset Analysis:")
    print(f"Shape: {master_dataset.shape}")
    
    # Show value counts for categorical columns
    categorical_cols = master_dataset.select_dtypes(include=['object']).columns
    
    for col in categorical_cols[:3]:  # Show first 3 categorical columns
        if master_dataset[col].nunique() < 20:  # Only for columns with reasonable unique values
            print(f"\n📋 Value counts for '{col}':")
            print(master_dataset[col].value_counts().head())
    
    # Numeric column analysis
    numeric_cols = master_dataset.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\n📊 Numeric columns summary:")
        display(master_dataset[numeric_cols].describe())

# Display final tables for manual inspection
print("\n" + "="*80)
print("FINAL EXTRACTED TABLES")
print("="*80)

for i, table in enumerate(final_tables, 1):
    print(f"\n📊 Table {i}:")
    display(table)
    
    if i < len(final_tables):  # Don't add separator after last table
        print("\n" + "-"*60)

print("\n🎉 PDF table extraction completed successfully!")
print("\n📁 Output files created:")
for file in csv_files:
    print(f"   📄 {file}")
print(f"   📊 {excel_file}")
print(f"   🌐 {html_file}")

## Next Steps

### Using with Your Own PDFs

To use this notebook with your own PDF files:

1. **Replace the sample PDF path** in the second cell with your PDF file path
2. **Run all cells** to extract and process your tables
3. **Review the validation results** to ensure data quality
4. **Export to your preferred format** using the generated files

### Customization Options

You can customize the extraction process by:
- **Adjusting similarity thresholds** for table continuity detection
- **Modifying cleaning rules** based on your data characteristics
- **Adding custom validation rules** for your specific use case
- **Customizing export formats** and styling

### Advanced Usage

For more advanced scenarios, consider:
- **Pre-processing PDFs** to improve table detection
- **Post-processing data** for specific business rules
- **Integrating with databases** for automatic data pipeline
- **Adding OCR capabilities** for scanned PDFs

### Troubleshooting

If you encounter issues:
1. **Check the validation results** for data quality issues
2. **Try different extraction methods** by modifying the parameters
3. **Examine the PDF structure** manually to understand layout complexity
4. **Use debug mode** for detailed extraction logs

Happy table extracting! 🚀